# 🌍 Galamsey Impact Tracker — SQL + Pandas + Plotly Capstone Project
**By Benedicta Bondzie (The Tech Diva)**  
**GitHub: TechDiva001**  

## Project Overview
This project is a data analysis capstone designed to track and analyze the environmental and public health impact of illegal small-scale gold mining (known in Ghana as **galamsey**) from 2020 to 2023.

Illegal mining in Ghana has led to severe pollution of freshwater bodies due to the discharge of toxic heavy metals (like mercury, lead, and cyanide) and high sediment loads (turbidity). This project uses a relational database to model:
1. **Regions** of Ghana and their demographics.
2. **Rivers** and their downstream connections.
3. **Water Quality Readings** over time at various stations.
4. **Mining Footprints** (land destroyed, chemicals used).
5. **Water Supply Intakes** and their risk exposure.
6. **Health Impacts** (toxic-linked disease cases and deaths).

## Technologies Used
*   **SQL (SQLite3)**: Database schema design, table relationships, and intermediate SQL analysis (joins, filters, aggregates).
*   **Pandas**: Data loading, verifying structure, cleaning missing values, converting data types, and adding safety flags.
*   **Plotly Express / Seaborn**: Inline interactive graphs for trends, distributions, correlations, and hierarchies.

---

# CELL 1 — IMPORTS and DATABASE CONNECTION

In [52]:
import sqlite3
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

conn = sqlite3.connect("galamsey.db")
conn.execute("PRAGMA foreign_keys = ON;")

print("✅ Database connected: galamsey.db created successfully")

✅ Database connected: galamsey.db created successfully


#Run this everytime you re-run the cell so every full re-run starts from a truly blank database

In [53]:
# Temporarily disable foreign key checks so DROP TABLE order doesn't matter
conn.execute("PRAGMA foreign_keys = OFF")

# DROP each table if it exists, so every full re-run starts from a truly blank database
tables_to_reset = [
    'regions', 'rivers', 'water_quality_readings',
    'mining_activity', 'water_supply_sources', 'health_impact'
]
for table in tables_to_reset:
    conn.execute(f"DROP TABLE IF EXISTS {table}")
conn.commit()
print("✅ All tables dropped, ready for a clean rebuild")


✅ All tables dropped, ready for a clean rebuild


## Part 1: Schema Design (DDL)
In this section, we create the database schema. The design consists of **6 tables** with defined primary keys, foreign keys to enforce relationships, and check constraints to prevent invalid data input (like negative hectares or invalid pH values).

### Table Relationships
*   `regions` is the parent table representing the 16 administrative regions of Ghana.
*   `rivers` originates in a region. It includes a `flows_into` text field describing where the river drains to trace contamination flow.
*   `water_quality_readings` belongs to a river and records physical and chemical metrics.
*   `mining_activity` records annual forest degradation and chemical use in regions.
*   `water_supply_sources` connects municipal water intakes to regions and rivers.
*   `health_impact` logs toxicity-linked clinical diagnoses and deaths by region and year.

# CELL 2 — CREATING ALL 6 TABLES

In [54]:
conn.executescript("""

    -- 1. Regions Table
    CREATE TABLE IF NOT EXISTS regions (
        region_id        INTEGER PRIMARY KEY AUTOINCREMENT,
        region_name      TEXT NOT NULL UNIQUE,
        capital          TEXT NOT NULL,
        population       INTEGER NOT NULL CHECK (population > 0),
        area_km2         REAL NOT NULL CHECK (area_km2 > 0),
        galamsey_hotspot INTEGER NOT NULL CHECK (galamsey_hotspot IN (0, 1))
    );

    -- 2. Rivers Table
    CREATE TABLE IF NOT EXISTS rivers (
        river_id        INTEGER PRIMARY KEY AUTOINCREMENT,
        river_name      TEXT NOT NULL ,
        region_id       INTEGER NOT NULL,
        length_km       REAL CHECK (length_km > 0),
        flows_into      TEXT, -- Name of the river/ocean it drains into
        is_contaminated INTEGER NOT NULL CHECK (is_contaminated IN (0, 1)),
        FOREIGN KEY (region_id) REFERENCES regions(region_id)
    );

    -- 3. Water Quality Readings Table
    CREATE TABLE IF NOT EXISTS water_quality_readings (
        reading_id      INTEGER PRIMARY KEY AUTOINCREMENT,
        river_id        INTEGER NOT NULL,
        reading_date    TEXT NOT NULL,
        ph_level        REAL CHECK (ph_level BETWEEN 0.0 AND 14.0),
        mercury_ug_L    REAL CHECK (mercury_ug_L >= 0.0),  -- micrograms per litre. WHO safe limit: 0.001
        turbidity_NTU   REAL CHECK (turbidity_NTU >= 0.0), -- NTU. Treatment plant max: 2000
        cyanide_mg_L    REAL CHECK (cyanide_mg_L >= 0.0), -- mg per litre. WHO safe limit: 0.07
        lead_ug_L       REAL CHECK (lead_ug_L >= 0.0), -- micrograms per litre. WHO safe limit: 10
        FOREIGN KEY (river_id) REFERENCES rivers(river_id)
    );

    -- 4. Mining Activity Table
    CREATE TABLE IF NOT EXISTS mining_activity (
        activity_id                  INTEGER PRIMARY KEY AUTOINCREMENT,
        region_id                    INTEGER NOT NULL,
        year                         INTEGER NOT NULL CHECK (year >= 2000),
        mining_type                  TEXT NOT NULL,
        hectares_destroyed           REAL NOT NULL CHECK (hectares_destroyed >= 0.0),
        num_sites                    INTEGER NOT NULL CHECK (num_sites >= 0),
        chemicals_used               TEXT NOT NULL CHECK (chemicals_used IN ('Mercury', 'Cyanide', 'Both', 'Mercury, Cyanide','None')),
        FOREIGN KEY (region_id) REFERENCES regions(region_id)
    );

    -- 5. Water Supply Sources Table
    CREATE TABLE IF NOT EXISTS water_supply_sources (
        source_id               INTEGER PRIMARY KEY AUTOINCREMENT,
        region_id               INTEGER NOT NULL,
        source_name             TEXT NOT NULL,
        source_type             TEXT NOT NULL CHECK (source_type IN ('River', 'Borehole', 'Reservoir', 'Dam')),
        is_contaminated         INTEGER NOT NULL CHECK (is_contaminated IN (0, 1)),
        communities_served      INTEGER NOT NULL CHECK (communities_served >= 0),
        population_at_risk      INTEGER NOT NULL CHECK (population_at_risk >= 0),
        treatment_plant_present INTEGER NOT NULL CHECK (treatment_plant_present IN (0, 1)),  -- 1 = YES, 0 = NO
        FOREIGN KEY (region_id) REFERENCES regions(region_id)
    );

    -- 6. Health Impact Table
    CREATE TABLE IF NOT EXISTS health_impact (
        impact_id         INTEGER PRIMARY KEY AUTOINCREMENT,
        region_id         INTEGER NOT NULL,
        year              INTEGER NOT NULL CHECK (year >= 2000),
        disease           TEXT NOT NULL,
        cases_reported    INTEGER NOT NULL CHECK (cases_reported >= 0),
        deaths            INTEGER NOT NULL CHECK (deaths >= 0),
        affected_children INTEGER NOT NULL CHECK (affected_children >= 0),
        cause             TEXT NOT NULL, -- Mercury, Cyanide, Lead, Contaminated River
        FOREIGN KEY (region_id) REFERENCES regions(region_id)
    );

""")
conn.commit()
print("✅ All 6 tables created successfully")

✅ All 6 tables created successfully


## Part 2: Data Seeding (DML)
We populate the database with seed data representing geographic, environmental, and public health metrics from 2020 to 2023.

### Data Characteristics
*   **Regions**: Active mining hubs (Ashanti, Western, Eastern, Ahafo, Central) are marked as hotspots (`galamsey_hotspot = 1`). Northern, Volta, and Upper regions are clean controls (`galamsey_hotspot = 0`).
*   **Rivers**: Polluted rivers (Birim, Pra, Offin, Ankobra) are marked as contaminated (`is_contaminated = 1`). Volta and Oti rivers are marked clean.
*   **Readings**: Exceedances of WHO safe limits (Mercury > 0.001 ug/L, Cyanide > 0.07 mg/L, Turbidity > 2000 NTU) are seeded in hotspot rivers, showing year-on-year increases.
*   **Mining Activities**: Focuses on hectares destroyed and site count by region.
*   **Health Impacts**: Attributes cases to specific toxic chemical causes (Mercury, Cyanide, Waterborne vectors).


# CELL 3 — INSERT DATA (SEEDS)


In [55]:

# 2. Seed regions table
regions_data = [

# (region_id, region_name, capital, population, area_km2, galamsey_hotspot)
    (1,  'Ashanti',      'Kumasi',          5440463, 24389.0, 1),
    (2,  'Western',      'Sekondi-Takoradi',2376021, 23921.0, 1),
    (3,  'Eastern',      'Koforidua',       2633154, 19323.0, 1),
    (4,  'Central',      'Cape Coast',      2563228, 9826.0,  1),
    (5,  'Ahafo',        'Goaso',           564891,  9399.0,  1),
    (6,  'Bono',         'Sunyani',         1256770, 12112.0, 1),
    (7,  'Bono East',    'Techiman',        1123845, 10723.0, 1),
    (8,  'Western North','Sefwi Wiawso',    763977,  12237.0, 1),
    (9,  'Northern',     'Tamale',          1820806, 70384.0, 0),
    (10, 'Savannah',     'Damongo',         423387,  37543.0, 0),
    (11, 'North East',   'Nalerigu',        527174,  15473.0, 0),
    (12, 'Upper East',   'Bolgatanga',      1015010, 8842.0,  0),
    (13, 'Upper West',   'Wa',              901226,  18476.0, 0),
    (14, 'Volta',        'Ho',              1475539, 20570.0, 0),
    (15, 'Oti',          'Dambai',          362089,  11731.0, 0),
    (16, 'Greater Accra','Accra',           5446789, 3245.0,  0),
]
conn.executemany("INSERT OR IGNORE INTO regions VALUES (?, ?, ?, ?, ?, ?)", regions_data)
conn.commit()
print("✅ 16 regions inserted")

✅ 16 regions inserted


# CELL 4 — INSERT: RIVERS

In [56]:
# 3. Seed rivers table

# is_contaminated: based on Water Resources Commission reports + research
# flows_into: where the river drains (shows river connectivity)


rivers_data = [

# (river_id, river_name, region_id, length_km, flows_into, is_contaminated)

    # ASHANTI (region 1)
    (1,  'Offin River',        1,  290.0, 'Pra River',           1),
    (2,  'Oda River',          1,  110.0, 'Pra River',           1),
    (3,  'Owabi River',        1,  60.0,  'Pra River',           1),

    # WESTERN (region 2)
    (4,  'Pra River',          2,  240.0, 'Atlantic Ocean',      1),
    (5,  'Ankobra River',      2,  209.0, 'Atlantic Ocean',      1),
    (6,  'Tano River',         2,  380.0, 'Atlantic Ocean',      1),

    # EASTERN (region 3)
    (7,  'Birim River',        3,  210.0, 'Pra River',           1),
    (8,  'Afram River',        3,  100.0, 'Lake Volta',          1),
    (9,  'Anum River',         3,  100.0, 'Volta River',         0),

    # CENTRAL (region 4)
    (10, 'Densu River',        4,  116.0, 'Atlantic Ocean',      1),
    (11, 'Ayensu River',       4,  90.0,  'Atlantic Ocean',      1),
    (12, 'Amisa River',        4,  55.0,  'Atlantic Ocean',      0),

    # AHAFO (region 5)
    (13, 'Bisi River',         5,  80.0,  'Subri River',         1),
    (14, 'Tano River (Ahafo)', 5,  380.0, 'Atlantic Ocean',      1),

    # BONO (region 6)
    (15, 'Tano River (Bono)',  6,  380.0, 'Atlantic Ocean',      1),
    (16, 'Subin River',        6,  70.0,  'Tano River',          1),

    # BONO EAST (region 7)
    (17, 'Afram River (BE)',   7,  100.0, 'Lake Volta',          0),
    (18, 'Pru River',          7,  150.0, 'Lake Volta',          0),

    # WESTERN NORTH (region 8)
    (19, 'Bia River',          8,  306.0, 'Atlantic Ocean',      1),
    (20, 'Tano River (WN)',    8,  380.0, 'Atlantic Ocean',      1),

    # NORTHERN (region 9)
    (21, 'White Volta',        9,  550.0, 'Lake Volta',          0),
    (22, 'Kulpawn River',      9,  200.0, 'White Volta',         0),
    (23, 'Nasia River',        9,  180.0, 'White Volta',         0),

    # SAVANNAH (region 10)
    (24, 'Black Volta',        10, 1352.0,'Lake Volta',          0),
    (25, 'Kulpawn River (S)',   10, 200.0, 'White Volta',         0),

    # NORTH EAST (region 11)
    (26, 'White Volta (NE)',   11, 550.0, 'Lake Volta',          0),
    (27, 'Asibelika River',    11, 80.0,  'White Volta',         0),

    # UPPER EAST (region 12)
    (28, 'Red Volta',          12, 160.0, 'White Volta',         0),
    (29, 'Nasia River (UE)',   12, 180.0, 'White Volta',         0),

    # UPPER WEST (region 13)
    (30, 'Black Volta (UW)',   13, 1352.0,'Lake Volta',          0),
    (31, 'Sissili River',      13, 250.0, 'White Volta',         0),

    # VOLTA (region 14)
    (32, 'Volta River',        14, 1600.0,'Atlantic Ocean',      0),
    (33, 'Todzie River',       14, 90.0,  'Lake Volta',          0),
    (34, 'Mo River',           14, 100.0, 'Oti River',           0),

    # OTI (region 15)
    (35, 'Oti River',          15, 370.0, 'Lake Volta',          0),
    (36, 'Daka River',         15, 150.0, 'Lake Volta',          0),

    # GREATER ACCRA (region 16)
    (37, 'Densu River (GA)',   16, 116.0, 'Atlantic Ocean',      1),
    (38, 'Sakumo Lagoon',      16, 30.0,  'Atlantic Ocean',      1),
]
conn.executemany("INSERT OR IGNORE INTO rivers VALUES (?, ?, ?, ?, ?, ?)", rivers_data)
conn.commit()
print("✅ Rivers inserted")

✅ Rivers inserted


# CELL 5 — INSERT: WATER QUALITY READINGS (2020–2026)

In [57]:
conn.executescript("DELETE FROM water_quality_readings; VACUUM;")

# 4. Seed water quality readings

# Real context:
# WHO Mercury safe limit: 0.001 µg/L
# WHO Lead safe limit: 10 µg/L
# WHO Cyanide safe limit: 0.07 mg/L
# Ghana Water Co treatment max turbidity: 2000 NTU
# Pra River reached 11,200 NTU in 2023 (WaterAid report)
# Mercury in hotspot rivers exceeds safe limits by 10–17x (research data)

readings_data = [

# (reading_id, river_id, date, ph, mercury_ug_L, turbidity_NTU, cyanide_mg_L, lead_ug_L)

       # OFFIN RIVER (Ashanti) — heavily mined, dredging
    (1,  1, '2020-03-15', 5.8, 0.0089, 3200.0,  0.12, 18.5),
    (2,  1, '2021-06-20', 5.6, 0.0112, 4100.0,  0.15, 22.1),
    (3,  1, '2022-09-10', 5.4, 0.0134, 5800.0,  0.18, 26.4),
    (4,  1, '2023-11-05', 5.2, 0.0158, 7200.0,  0.22, 31.2),
    (5, 1, '2024-02-10', 5.0, 0.0182, 9100.0,  0.26, 36.8),
    (6, 1, '2025-05-15', 4.8, 0.0209, 11400.0, 0.31, 42.1),
    (7, 1, '2026-01-20', 4.6, 0.0238, 13800.0, 0.36, 48.3),



    # ODA RIVER (Ashanti)
    (8,  2, '2020-04-10', 5.9, 0.0082, 2900.0,  0.11, 16.2),
    (9,  2, '2021-07-14', 5.7, 0.0101, 3800.0,  0.13, 19.8),
    (10,  2, '2022-10-22', 5.5, 0.0123, 5200.0,  0.16, 24.1),
    (11,  2, '2023-12-18', 5.3, 0.0148, 6900.0,  0.20, 29.3),
    (12, 2, '2024-03-12', 5.1, 0.0171, 8600.0,  0.23, 33.9),
    (13, 2, '2025-06-18', 4.9, 0.0196, 10800.0, 0.28, 39.2),
    (14, 2, '2026-02-22', 4.7, 0.0224, 13100.0, 0.33, 45.4),

    # OWABI RIVER (Ashanti) — water supply for Kumasi
    (15,  3, '2020-05-20', 6.1, 0.0071, 2100.0,  0.09, 14.3),
    (16, 3, '2021-08-25', 5.9, 0.0088, 2900.0,  0.11, 17.6),
    (17, 3, '2022-11-12', 5.7, 0.0108, 4100.0,  0.14, 21.4),
    (18, 3, '2023-02-28', 5.5, 0.0131, 5600.0,  0.17, 26.8),
    (19, 3, '2024-04-05', 5.3, 0.0151, 7100.0,  0.20, 31.2),
    (20, 3, '2025-07-10', 5.1, 0.0174, 8900.0,  0.24, 36.5),
    (21, 3, '2026-03-14', 4.9, 0.0199, 10800.0, 0.28, 42.1),

    # PRA RIVER (Western) — most polluted, reaches 11,200 NTU
    (22, 4, '2020-02-11', 5.9, 0.0095, 4500.0,  0.13, 19.1),
    (23, 4, '2021-05-19', 5.7, 0.0118, 6200.0,  0.17, 23.5),
    (24, 4, '2022-08-14', 5.5, 0.0145, 8900.0,  0.21, 28.9),
    (25, 4, '2023-10-30', 5.1, 0.0172, 11200.0, 0.25, 34.7),
    (26, 4, '2024-01-15', 4.9, 0.0198, 14000.0, 0.29, 39.8),
    (27, 4, '2025-04-20', 4.7, 0.0228, 16500.0, 0.34, 45.9),
    (28, 4, '2026-01-25', 4.5, 0.0261, 18900.0, 0.39, 52.4),

    # ANKOBRA RIVER (Western)
    (29, 5, '2020-04-22', 6.1, 0.0078, 2800.0,  0.09, 15.2),
    (30, 5, '2021-07-18', 5.9, 0.0098, 3900.0,  0.12, 18.9),
    (31, 5, '2022-10-05', 5.7, 0.0121, 5100.0,  0.16, 23.4),
    (32, 5, '2023-12-12', 5.4, 0.0149, 6800.0,  0.20, 28.7),
    (33, 5, '2024-03-18', 5.2, 0.0171, 8500.0,  0.23, 33.1),
    (34, 5, '2025-06-23', 5.0, 0.0197, 10600.0, 0.27, 38.4),
    (35, 5, '2026-02-27', 4.8, 0.0226, 12900.0, 0.31, 44.2),

    # TANO RIVER (Western)
    (36, 6, '2020-06-15', 6.2, 0.0065, 2200.0,  0.08, 13.1),
    (37, 6, '2021-09-20', 6.0, 0.0082, 3100.0,  0.10, 16.4),
    (38, 6, '2022-12-08', 5.8, 0.0101, 4400.0,  0.13, 20.2),
    (39, 6, '2023-03-25', 5.5, 0.0124, 6000.0,  0.17, 25.1),
    (40, 6, '2024-05-01', 5.3, 0.0143, 7600.0,  0.20, 29.3),
    (41, 6, '2025-08-06', 5.1, 0.0165, 9500.0,  0.24, 34.1),
    (42, 6, '2026-04-10', 4.9, 0.0189, 11500.0, 0.27, 39.4),

    # BIRIM RIVER (Eastern) — diamond mining hub
    (43, 7, '2020-01-30', 6.0, 0.0082, 3100.0,  0.10, 16.7),
    (44, 7, '2021-04-25', 5.8, 0.0105, 4400.0,  0.14, 20.9),
    (45, 7, '2022-07-08', 5.6, 0.0128, 6100.0,  0.18, 25.8),
    (46, 7, '2023-09-22', 5.3, 0.0155, 8300.0,  0.23, 31.4),
    (47, 7, '2024-01-08', 5.1, 0.0179, 10200.0, 0.27, 36.1),
    (48, 7, '2025-04-13', 4.9, 0.0206, 12700.0, 0.31, 41.8),
    (49, 7, '2026-01-17', 4.7, 0.0236, 15400.0, 0.36, 48.2),

    # AFRAM RIVER (Eastern)
    (50, 8, '2020-03-05', 6.3, 0.0061, 1800.0,  0.07, 12.3),
    (51, 8, '2021-06-10', 6.1, 0.0078, 2500.0,  0.09, 15.6),
    (52, 8, '2022-09-18', 5.9, 0.0097, 3500.0,  0.12, 19.2),
    (53, 8, '2023-11-30', 5.6, 0.0118, 4800.0,  0.15, 23.8),
    (54, 8, '2024-02-14', 5.4, 0.0136, 6100.0,  0.18, 27.5),
    (55, 8, '2025-05-19', 5.2, 0.0157, 7600.0,  0.21, 31.9),
    (56, 8, '2026-03-23', 5.0, 0.0180, 9200.0,  0.24, 37.0),

    # DENSU RIVER (Central) — supplies water to Accra area
    (57, 10, '2020-05-14', 6.3, 0.0065, 1900.0,  0.07, 13.5),
    (58, 10, '2021-08-09', 6.1, 0.0081, 2700.0,  0.09, 16.8),
    (59, 10, '2022-11-16', 5.9, 0.0099, 3800.0,  0.12, 20.5),
    (60, 10, '2023-01-28', 5.6, 0.0118, 5200.0,  0.15, 25.1),
    (61, 10, '2024-03-20', 5.4, 0.0136, 6500.0,  0.18, 29.1),
    (62, 10, '2025-06-25', 5.2, 0.0157, 8100.0,  0.21, 33.8),
    (63, 10, '2026-04-29', 5.0, 0.0180, 9800.0,  0.24, 39.1),

    # BISI RIVER (Ahafo) — heavily polluted, flows into Subri then Tano
    (64, 13, '2020-02-18', 5.7, 0.0091, 3400.0,  0.12, 17.8),
    (65, 13, '2021-05-23', 5.5, 0.0115, 4800.0,  0.15, 22.3),
    (66, 13, '2022-08-30', 5.3, 0.0141, 6700.0,  0.19, 27.6),
    (67, 13, '2023-10-14', 5.0, 0.0169, 9100.0,  0.24, 33.9),
    (68, 13, '2024-01-21', 4.8, 0.0194, 11800.0, 0.28, 38.9),
    (69, 13, '2025-04-26', 4.6, 0.0224, 14700.0, 0.33, 45.1),
    (70, 13, '2026-02-01', 4.4, 0.0256, 17800.0, 0.38, 52.1),

    # TANO RIVER AHAFO
    (71, 14, '2020-03-22', 5.9, 0.0079, 2800.0,  0.10, 15.4),
    (72, 14, '2021-06-27', 5.7, 0.0099, 3900.0,  0.12, 19.1),
    (73, 14, '2022-09-14', 5.5, 0.0122, 5400.0,  0.16, 23.8),
    (74, 14, '2023-11-28', 5.2, 0.0148, 7300.0,  0.20, 29.5),
    (75, 14, '2024-02-24', 5.0, 0.0170, 9200.0,  0.24, 34.0),
    (76, 14, '2025-05-30', 4.8, 0.0196, 11500.0, 0.28, 39.5),
    (77, 14, '2026-03-05', 4.6, 0.0225, 13900.0, 0.32, 45.7),

    # TANO RIVER BONO
    (78, 15, '2020-04-10', 6.0, 0.0072, 2500.0,  0.09, 14.2),
    (79, 15, '2021-07-15', 5.8, 0.0091, 3500.0,  0.11, 17.8),
    (80, 15, '2022-10-20', 5.6, 0.0112, 4800.0,  0.14, 22.1),
    (81, 15, '2023-01-05', 5.3, 0.0136, 6500.0,  0.18, 27.4),
    (82, 15, '2024-03-27', 5.1, 0.0157, 8300.0,  0.21, 31.6),
    (83, 15, '2025-07-02', 4.9, 0.0181, 10400.0, 0.25, 36.7),
    (84, 15, '2026-04-06', 4.7, 0.0208, 12600.0, 0.29, 42.5),

    # BIA RIVER (Western North)
    (85, 19, '2020-05-08', 6.0, 0.0076, 2600.0,  0.09, 14.9),
    (86, 19, '2021-08-13', 5.8, 0.0095, 3600.0,  0.12, 18.7),
    (87, 19, '2022-11-19', 5.6, 0.0117, 5000.0,  0.15, 23.1),
    (88, 19, '2023-02-24', 5.3, 0.0142, 6800.0,  0.19, 28.6),
    (89, 19, '2024-04-03', 5.1, 0.0163, 8700.0,  0.22, 32.9),
    (90, 19, '2025-07-09', 4.9, 0.0188, 10900.0, 0.26, 38.1),
    (91, 19, '2026-05-13', 4.7, 0.0216, 13200.0, 0.30, 44.1),

    # WHITE VOLTA (Northern) — mostly clean, no major galamsey
    (92, 21, '2020-06-01', 6.9, 0.0028, 720.0,   0.02, 5.1),
    (93, 21, '2021-09-05', 6.8, 0.0035, 980.0,   0.03, 6.3),
    (94, 21, '2022-12-10', 6.7, 0.0043, 1250.0,  0.04, 7.8),
    (95, 21, '2023-03-15', 6.6, 0.0052, 1580.0,  0.05, 9.2),
    (96, 21, '2024-06-10', 6.5, 0.0062, 1900.0,  0.06, 10.8),
    (97, 21, '2025-09-15', 6.4, 0.0073, 2300.0,  0.07, 12.4),
    (98, 21, '2026-06-20', 6.3, 0.0085, 2800.0,  0.08, 14.2),

    # BLACK VOLTA (Savannah) — largely clean
    (99, 24, '2020-07-01', 7.1, 0.0021, 580.0,   0.02, 4.2),
    (100, 24, '2021-10-06', 7.0, 0.0028, 790.0,   0.02, 5.1),
    (101, 24, '2022-01-12', 6.9, 0.0034, 1010.0,  0.03, 6.4),
    (102, 24, '2023-04-18', 6.8, 0.0041, 1280.0,  0.04, 7.8),
    (103, 24, '2024-07-12', 6.7, 0.0049, 1580.0,  0.05, 9.4),
    (104, 24, '2025-10-17', 6.6, 0.0058, 1920.0,  0.06, 11.1),
    (105, 24, '2026-06-22', 6.5, 0.0068, 2340.0,  0.07, 13.0),
    # OTI RIVER (Oti region) — clean, no mining
    (106, 35, '2020-08-01', 7.2, 0.0018, 450.0,   0.01, 3.2),
    (107, 35, '2021-11-06', 7.1, 0.0023, 610.0,   0.02, 4.1),
    (108, 35, '2022-02-11', 7.0, 0.0029, 820.0,   0.02, 5.3),
    (109, 35, '2023-05-17', 6.9, 0.0036, 1050.0,  0.03, 6.7),
    (110, 35, '2024-08-14', 7.0, 0.0043, 1280.0,  0.03, 7.8),
    (111, 35, '2025-11-19', 6.9, 0.0051, 1560.0,  0.04, 9.1),
    (112, 35, '2026-05-24', 6.8, 0.0060, 1890.0,  0.04, 10.6),

    # VOLTA RIVER (Volta region)
    (113, 32, '2020-09-01', 7.0, 0.0025, 510.0,   0.02, 4.5),
    (114, 32, '2021-12-06', 6.9, 0.0031, 690.0,   0.02, 5.6),
    (115, 32, '2022-03-11', 6.8, 0.0038, 920.0,   0.03, 7.1),
    (116, 32, '2023-06-17', 6.7, 0.0046, 1180.0,  0.04, 8.9),
    (117, 32, '2024-09-16', 6.6, 0.0055, 1420.0,  0.04, 10.4),
    (118, 32, '2025-12-21', 6.5, 0.0064, 1730.0,  0.05, 12.0),
    (119, 32, '2026-06-25', 6.4, 0.0075, 2100.0,  0.06, 13.9),

    # DENSU — GREATER ACCRA (downstream from Central)
    (120, 37, '2020-10-01', 6.2, 0.0070, 2100.0,  0.08, 14.1),
    (121, 37, '2021-01-06', 6.0, 0.0088, 2900.0,  0.10, 17.5),
    (122, 37, '2022-04-11', 5.8, 0.0108, 4000.0,  0.13, 21.4),
    (123, 37, '2023-07-17', 5.5, 0.0131, 5500.0,  0.16, 26.2),
    (127, 32, '2024-09-16', 6.6, 0.0055, 1420.0,  0.04, 10.4),
    (128, 32, '2025-12-21', 6.5, 0.0064, 1730.0,  0.05, 12.0),
    (129, 32, '2026-06-25', 6.4, 0.0075, 2100.0,  0.06, 13.9),
]
conn.executemany("INSERT INTO water_quality_readings VALUES (?, ?, ?, ?, ?, ?, ?, ?)", readings_data)
conn.commit()

# CELL 6 — INSERT: MINING ACTIVITY (2020–2026)

In [58]:
# Gold hit $3,000/gram late 2024 — caused massive surge in galamsey
# (Al Jazeera, January 2025)

mining_data = [
    # (activity_id, region_id, year, mining_type, hectares_destroyed, num_sites, chemicals_used)

    # ASHANTI
    (1,  1, 2020, 'Dredging',       320.0,  45,  'Mercury'),
    (2,  1, 2021, 'Dredging',       410.0,  58,  'Mercury'),
    (3,  1, 2022, 'Washing Plant',  520.0,  71,  'Mercury, Cyanide'),
    (4,  1, 2023, 'Washing Plant',  640.0,  89,  'Mercury, Cyanide'),
    (5,  1, 2024, 'Dredging',       800.0,  112, 'Mercury, Cyanide'),
    (6,  1, 2025, 'Washing Plant',  980.0,  138, 'Mercury, Cyanide'),
    (7,  1, 2026, 'Washing Plant',  1150.0, 162, 'Mercury, Cyanide'),

    # WESTERN
    (8,  2, 2020, 'Mill House',     280.0,  38,  'Mercury'),
    (9,  2, 2021, 'Mill House',     370.0,  51,  'Mercury, Cyanide'),
    (10, 2, 2022, 'Dredging',       480.0,  64,  'Mercury, Cyanide'),
    (11, 2, 2023, 'Dredging',       590.0,  78,  'Mercury, Cyanide'),
    (12, 2, 2024, 'Dredging',       730.0,  97,  'Mercury, Cyanide'),
    (13, 2, 2025, 'Dredging',       890.0,  119, 'Mercury, Cyanide'),
    (14, 2, 2026, 'Mill House',     1050.0, 140, 'Mercury, Cyanide'),

    # EASTERN
    (15, 3, 2020, 'Washing Board',  190.0,  29,  'Mercury'),
    (16, 3, 2021, 'Washing Board',  250.0,  38,  'Mercury'),
    (17, 3, 2022, 'Chamfi',         310.0,  47,  'Mercury, Cyanide'),
    (18, 3, 2023, 'Chamfi',         390.0,  56,  'Mercury, Cyanide'),
    (19, 3, 2024, 'Chamfi',         480.0,  69,  'Mercury, Cyanide'),
    (20, 3, 2025, 'Chamfi',         590.0,  85,  'Mercury, Cyanide'),
    (21, 3, 2026, 'Dredging',       700.0,  101, 'Mercury, Cyanide'),

    # CENTRAL
    (22, 4, 2020, 'Dig and Wash',   120.0,  18,  'Mercury'),
    (23, 4, 2021, 'Dig and Wash',   160.0,  24,  'Mercury'),
    (24, 4, 2022, 'Washing Plant',  210.0,  31,  'Mercury, Cyanide'),
    (25, 4, 2023, 'Washing Plant',  270.0,  39,  'Mercury, Cyanide'),
    (26, 4, 2024, 'Washing Plant',  330.0,  48,  'Mercury, Cyanide'),
    (27, 4, 2025, 'Washing Plant',  405.0,  59,  'Mercury, Cyanide'),
    (28, 4, 2026, 'Washing Plant',  480.0,  70,  'Mercury, Cyanide'),

    # AHAFO
    (29, 5, 2020, 'Dredging',       210.0,  31,  'Mercury'),
    (30, 5, 2021, 'Dredging',       280.0,  41,  'Mercury, Cyanide'),
    (31, 5, 2022, 'Mill House',     360.0,  53,  'Mercury, Cyanide'),
    (32, 5, 2023, 'Mill House',     450.0,  67,  'Mercury, Cyanide'),
    (33, 5, 2024, 'Dredging',       560.0,  83,  'Mercury, Cyanide'),
    (34, 5, 2025, 'Mill House',     690.0,  102, 'Mercury, Cyanide'),
    (35, 5, 2026, 'Mill House',     820.0,  122, 'Mercury, Cyanide'),

    # BONO
    (36, 6, 2020, 'Washing Board',  140.0,  21,  'Mercury'),
    (37, 6, 2021, 'Washing Board',  185.0,  28,  'Mercury'),
    (38, 6, 2022, 'Dig and Wash',   230.0,  35,  'Mercury'),
    (39, 6, 2023, 'Washing Plant',  290.0,  44,  'Mercury, Cyanide'),
    (40, 6, 2024, 'Washing Plant',  360.0,  54,  'Mercury, Cyanide'),
    (41, 6, 2025, 'Washing Plant',  440.0,  66,  'Mercury, Cyanide'),
    (42, 6, 2026, 'Dredging',       520.0,  79,  'Mercury, Cyanide'),

    # BONO EAST
    (43, 7, 2020, 'Panning',         40.0,  6,   'None'),
    (44, 7, 2021, 'Panning',         58.0,  9,   'None'),
    (45, 7, 2022, 'Dig and Wash',    80.0,  13,  'Mercury'),
    (46, 7, 2023, 'Dig and Wash',   105.0,  17,  'Mercury'),
    (47, 7, 2024, 'Dig and Wash',   135.0,  22,  'Mercury'),
    (48, 7, 2025, 'Washing Plant',  175.0,  29,  'Mercury, Cyanide'),
    (49, 7, 2026, 'Washing Plant',  220.0,  36,  'Mercury, Cyanide'),

    # WESTERN NORTH
    (50, 8, 2020, 'Dredging',       175.0,  26,  'Mercury'),
    (51, 8, 2021, 'Mill House',     225.0,  34,  'Mercury, Cyanide'),
    (52, 8, 2022, 'Mill House',     295.0,  44,  'Mercury, Cyanide'),
    (53, 8, 2023, 'Dredging',       375.0,  56,  'Mercury, Cyanide'),
    (54, 8, 2024, 'Dredging',       465.0,  69,  'Mercury, Cyanide'),
    (55, 8, 2025, 'Dredging',       570.0,  85,  'Mercury, Cyanide'),
    (56, 8, 2026, 'Mill House',     675.0,  101, 'Mercury, Cyanide'),

    # SAVANNAH — emerging galamsey
    (57, 10, 2022, 'Panning',        28.0,  5,   'None'),
    (58, 10, 2023, 'Dig and Wash',   41.0,  7,   'Mercury'),
    (59, 10, 2024, 'Dig and Wash',   58.0,  10,  'Mercury'),
    (60, 10, 2025, 'Washing Board',  78.0,  14,  'Mercury'),
    (61, 10, 2026, 'Dig and Wash',  102.0,  18,  'Mercury'),
]

conn.executemany("INSERT OR IGNORE INTO mining_activity VALUES (?, ?, ?, ?, ?, ?, ?)", mining_data)
conn.commit()
print("✅ Mining activity inserted (2020–2026)")

✅ Mining activity inserted (2020–2026)


# CELL 7 — INSERT: WATER SUPPLY SOURCES

In [60]:
water_supply_data = [
    # (source_id, region_id, source_name, source_type,
    #  is_contaminated, communities_served, population_at_risk, treatment_plant_present)

    (1,  1,  'Owabi Reservoir',               'Reservoir', 1, 18, 890000,  1),
    (2,  1,  'Offin River Intake',             'River',     1, 12, 420000,  1),
    (3,  2,  'Pra River Treatment Plant',      'River',     1, 15, 1100000, 1),
    (4,  2,  'Ankobra Intake',                 'River',     1, 9,  380000,  0),
    (5,  3,  'Birim River Intake',             'River',     1, 11, 460000,  1),
    (6,  3,  'Afram Borehole Network',         'Borehole',  0, 7,  85000,   0),
    (7,  4,  'Densu Dam',                      'Dam',       1, 13, 750000,  1),
    (8,  4,  'Ayensu River Intake',            'River',     1, 8,  310000,  0),
    (9,  5,  'Bisi River Supply',              'River',     1, 6,  95000,   0),
    (10, 5,  'Tano Acherensua Treatment',      'River',     1, 9,  145000,  1),
    (11, 6,  'Tano Abesim Treatment Plant',    'River',     1, 11, 280000,  1),
    (12, 6,  'Subin River Intake',             'River',     1, 7,  165000,  0),
    (13, 7,  'Tano Tanoso Treatment Plant',    'River',     1, 8,  210000,  1),
    (14, 7,  'Pru Borehole System',            'Borehole',  0, 12, 95000,   0),
    (15, 8,  'Bia River Intake',               'River',     1, 8,  180000,  0),
    (16, 8,  'Tano Western North Intake',      'River',     1, 6,  130000,  0),
    (17, 9,  'White Volta Treatment',          'River',     0, 14, 340000,  1),
    (18, 9,  'Tamale Borehole System',         'Borehole',  0, 18, 520000,  0),
    (19, 10, 'Black Volta Intake',             'River',     0, 7,  95000,   0),
    (20, 11, 'White Volta NE Intake',          'River',     0, 9,  120000,  0),
    (21, 12, 'Red Volta Treatment',            'River',     0, 11, 210000,  1),
    (22, 12, 'Bolgatanga Borehole Network',    'Borehole',  0, 14, 185000,  0),
    (23, 13, 'Black Volta Wa Treatment',       'River',     0, 8,  165000,  1),
    (24, 13, 'Sissili Borehole System',        'Borehole',  0, 12, 140000,  0),
    (25, 14, 'Volta River Intake Ho',          'River',     0, 16, 480000,  1),
    (26, 14, 'Todzie River Intake',            'River',     0, 8,  190000,  0),
    (27, 15, 'Oti River Intake',               'River',     0, 6,  95000,   0),
    (28, 15, 'Daka Borehole System',           'Borehole',  0, 9,  75000,   0),
    (29, 16, 'Weija Dam (Densu)',              'Dam',       1, 22, 2100000, 1),
    (30, 16, 'Kpong Water Treatment',          'River',     0, 18, 1800000, 1),
]

conn.executemany("INSERT OR IGNORE INTO water_supply_sources VALUES (?, ?, ?, ?, ?, ?, ?, ?)", water_supply_data)
conn.commit()
print("✅ Water supply sources inserted")

✅ Water supply sources inserted


# CELL 8 — INSERT: HEALTH IMPACT (2020–2026)

In [61]:

# Real sources:
# 2024: Respiratory cases hit 560/100,000 in galamsey regions (NCCE Ghana)
# 2024: Galamsey linked to kidney failures, birth defects, cancers (ISSA Africa)
# 2025–2026: projected from confirmed accelerating trend

health_data = [
    # (impact_id, region_id, year, disease, cases, deaths, children_affected, cause)

    # ASHANTI
    (1,  1, 2020, 'Chronic Kidney Disease', 320, 12, 45,  'Mercury'),
    (2,  1, 2021, 'Chronic Kidney Disease', 410, 18, 62,  'Mercury'),
    (3,  1, 2022, 'Neurological Disorder',  180,  6, 38,  'Mercury'),
    (4,  1, 2022, 'Mercury Poisoning',      240,  9, 51,  'Mercury'),
    (5,  1, 2023, 'Chronic Kidney Disease', 520, 22, 79,  'Mercury'),
    (6,  1, 2023, 'Birth Defects',           42,  3, 42,  'Cyanide'),
    (7,  1, 2024, 'Chronic Kidney Disease', 650, 28, 98,  'Mercury'),
    (8,  1, 2024, 'Respiratory Disease',    580, 14, 88,  'Mercury'),
    (9,  1, 2024, 'Birth Defects',           56,  4, 56,  'Cyanide'),
    (10, 1, 2025, 'Chronic Kidney Disease', 810, 35, 122, 'Mercury'),
    (11, 1, 2025, 'Respiratory Disease',    710, 18, 108, 'Mercury'),
    (12, 1, 2026, 'Chronic Kidney Disease', 980, 43, 148, 'Mercury'),
    (13, 1, 2026, 'Respiratory Disease',    860, 22, 131, 'Mercury'),

    # WESTERN
    (14, 2, 2020, 'Chronic Kidney Disease', 280, 10, 39,  'Mercury'),
    (15, 2, 2021, 'Respiratory Disease',    350,  8, 55,  'Mercury'),
    (16, 2, 2022, 'Mercury Poisoning',      290, 11, 48,  'Mercury'),
    (17, 2, 2023, 'Chronic Kidney Disease', 420, 16, 71,  'Mercury'),
    (18, 2, 2023, 'Birth Defects',           38,  2, 38,  'Cyanide'),
    (19, 2, 2024, 'Chronic Kidney Disease', 530, 20, 89,  'Mercury'),
    (20, 2, 2024, 'Respiratory Disease',    490, 12, 79,  'Mercury'),
    (21, 2, 2025, 'Chronic Kidney Disease', 660, 26, 111, 'Mercury'),
    (22, 2, 2025, 'Birth Defects',           51,  3, 51,  'Cyanide'),
    (23, 2, 2026, 'Chronic Kidney Disease', 800, 32, 135, 'Mercury'),
    (24, 2, 2026, 'Respiratory Disease',    620, 16, 101, 'Mercury'),

    # EASTERN
    (25, 3, 2020, 'Waterborne Disease',     190,  5, 31,  'Contaminated Water'),
    (26, 3, 2021, 'Chronic Kidney Disease', 240,  9, 42,  'Mercury'),
    (27, 3, 2022, 'Neurological Disorder',  160,  4, 29,  'Mercury'),
    (28, 3, 2023, 'Mercury Poisoning',      210,  8, 44,  'Mercury'),
    (29, 3, 2023, 'Chronic Kidney Disease', 310, 12, 58,  'Mercury'),
    (30, 3, 2024, 'Chronic Kidney Disease', 390, 15, 73,  'Mercury'),
    (31, 3, 2024, 'Neurological Disorder',  200,  5, 36,  'Mercury'),
    (32, 3, 2025, 'Chronic Kidney Disease', 490, 19, 91,  'Mercury'),
    (33, 3, 2026, 'Chronic Kidney Disease', 595, 24, 111, 'Mercury'),
    (34, 3, 2026, 'Respiratory Disease',    310,  8, 51,  'Mercury'),

    # CENTRAL
    (35, 4, 2020, 'Waterborne Disease',     140,  3, 22,  'Contaminated Water'),
    (36, 4, 2021, 'Respiratory Disease',    180,  5, 31,  'Mercury'),
    (37, 4, 2022, 'Chronic Kidney Disease', 160,  6, 28,  'Mercury'),
    (38, 4, 2023, 'Chronic Kidney Disease', 220,  8, 39,  'Mercury'),
    (39, 4, 2024, 'Chronic Kidney Disease', 275, 10, 49,  'Mercury'),
    (40, 4, 2025, 'Chronic Kidney Disease', 345, 13, 61,  'Mercury'),
    (41, 4, 2026, 'Chronic Kidney Disease', 415, 16, 74,  'Mercury'),

    # AHAFO
    (42, 5, 2020, 'Waterborne Disease',     110,  3, 19,  'Contaminated Water'),
    (43, 5, 2021, 'Chronic Kidney Disease', 145,  5, 26,  'Mercury'),
    (44, 5, 2022, 'Mercury Poisoning',      180,  7, 34,  'Mercury'),
    (45, 5, 2023, 'Chronic Kidney Disease', 230,  9, 45,  'Mercury'),
    (46, 5, 2024, 'Chronic Kidney Disease', 290, 11, 57,  'Mercury'),
    (47, 5, 2024, 'Mercury Poisoning',      225,  9, 45,  'Mercury'),
    (48, 5, 2025, 'Chronic Kidney Disease', 365, 14, 71,  'Mercury'),
    (49, 5, 2026, 'Chronic Kidney Disease', 445, 18, 87,  'Mercury'),

    # BONO
    (50, 6, 2020, 'Waterborne Disease',      90,  2, 15,  'Contaminated Water'),
    (51, 6, 2021, 'Waterborne Disease',     120,  3, 21,  'Contaminated Water'),
    (52, 6, 2022, 'Chronic Kidney Disease', 140,  5, 27,  'Mercury'),
    (53, 6, 2023, 'Chronic Kidney Disease', 185,  7, 36,  'Mercury'),
    (54, 6, 2024, 'Chronic Kidney Disease', 235,  9, 46,  'Mercury'),
    (55, 6, 2025, 'Chronic Kidney Disease', 295, 11, 58,  'Mercury'),
    (56, 6, 2026, 'Chronic Kidney Disease', 360, 14, 70,  'Mercury'),

    # BONO EAST
    (57, 7, 2022, 'Waterborne Disease',      60,  1, 11,  'Contaminated Water'),
    (58, 7, 2023, 'Waterborne Disease',      85,  2, 16,  'Contaminated Water'),
    (59, 7, 2024, 'Waterborne Disease',     110,  3, 20,  'Contaminated Water'),
    (60, 7, 2025, 'Chronic Kidney Disease', 140,  5, 28,  'Mercury'),
    (61, 7, 2026, 'Chronic Kidney Disease', 180,  7, 36,  'Mercury'),

    # WESTERN NORTH
    (62, 8, 2020, 'Waterborne Disease',      95,  2, 16,  'Contaminated Water'),
    (63, 8, 2021, 'Chronic Kidney Disease', 125,  4, 23,  'Mercury'),
    (64, 8, 2022, 'Mercury Poisoning',      155,  6, 31,  'Mercury'),
    (65, 8, 2023, 'Chronic Kidney Disease', 200,  8, 41,  'Mercury'),
    (66, 8, 2024, 'Chronic Kidney Disease', 250, 10, 51,  'Mercury'),
    (67, 8, 2025, 'Chronic Kidney Disease', 315, 13, 64,  'Mercury'),
    (68, 8, 2026, 'Chronic Kidney Disease', 385, 16, 78,  'Mercury'),

    # NORTHERN
    (69, 9, 2020, 'Waterborne Disease',      75,  2, 14,  'Contaminated Water'),
    (70, 9, 2021, 'Waterborne Disease',      85,  2, 16,  'Contaminated Water'),
    (71, 9, 2022, 'Respiratory Disease',     95,  3, 18,  'Mercury'),
    (72, 9, 2023, 'Respiratory Disease',    110,  3, 21,  'Mercury'),
    (73, 9, 2024, 'Respiratory Disease',    125,  4, 24,  'Mercury'),
    (74, 9, 2025, 'Respiratory Disease',    145,  4, 28,  'Mercury'),
    (75, 9, 2026, 'Respiratory Disease',    168,  5, 32,  'Mercury'),

    # GREATER ACCRA — downstream Densu + Weija Dam contamination
    (76, 16, 2020, 'Waterborne Disease',    160,  4, 28,  'Contaminated Water'),
    (77, 16, 2021, 'Waterborne Disease',    210,  5, 38,  'Contaminated Water'),
    (78, 16, 2022, 'Chronic Kidney Disease',190,  7, 35,  'Mercury'),
    (79, 16, 2023, 'Chronic Kidney Disease',250,  9, 48,  'Mercury'),
    (80, 16, 2024, 'Chronic Kidney Disease',315, 11, 61,  'Mercury'),
    (81, 16, 2024, 'Waterborne Disease',    280,  7, 52,  'Contaminated Water'),
    (82, 16, 2025, 'Chronic Kidney Disease',395, 14, 76,  'Mercury'),
    (83, 16, 2026, 'Chronic Kidney Disease',480, 18, 93,  'Mercury'),
]

conn.executemany("INSERT OR IGNORE INTO health_impact VALUES (?, ?, ?, ?, ?, ?, ?, ?)", health_data)
conn.commit()
print("✅ Health impact data inserted (2020–2026)")

✅ Health impact data inserted (2020–2026)


# CELL 9 — VERIFY ALL TABLES

In [62]:

tables = ['regions','rivers','water_quality_readings',
          'mining_activity','water_supply_sources','health_impact']

print("=== TABLE ROW COUNTS ===")
for t in tables:
    df = pd.read_sql(f"SELECT * FROM {t}", conn)
    print(f"\n{t}:\n")
    print(df)


=== TABLE ROW COUNTS ===

regions:

    region_id    region_name           capital  population  area_km2  \
0           1        Ashanti            Kumasi     5440463   24389.0   
1           2        Western  Sekondi-Takoradi     2376021   23921.0   
2           3        Eastern         Koforidua     2633154   19323.0   
3           4        Central        Cape Coast     2563228    9826.0   
4           5          Ahafo             Goaso      564891    9399.0   
5           6           Bono           Sunyani     1256770   12112.0   
6           7      Bono East          Techiman     1123845   10723.0   
7           8  Western North      Sefwi Wiawso      763977   12237.0   
8           9       Northern            Tamale     1820806   70384.0   
9          10       Savannah           Damongo      423387   37543.0   
10         11     North East          Nalerigu      527174   15473.0   
11         12     Upper East        Bolgatanga     1015010    8842.0   
12         13     Upper West

In [63]:
print("\n=== MISSING VALUES — WATER QUALITY ===")
for t in tables:
    df = pd.read_sql(f"SELECT * FROM {t}", conn)
    print(f"\n{t}:\n")
    print(df.isnull().sum())


=== MISSING VALUES — WATER QUALITY ===

regions:

region_id           0
region_name         0
capital             0
population          0
area_km2            0
galamsey_hotspot    0
dtype: int64

rivers:

river_id           0
river_name         0
region_id          0
length_km          0
flows_into         0
is_contaminated    0
dtype: int64

water_quality_readings:

reading_id       0
river_id         0
reading_date     0
ph_level         0
mercury_ug_L     0
turbidity_NTU    0
cyanide_mg_L     0
lead_ug_L        0
dtype: int64

mining_activity:

activity_id           0
region_id             0
year                  0
mining_type           0
hectares_destroyed    0
num_sites             0
chemicals_used        0
dtype: int64

water_supply_sources:

source_id                  0
region_id                  0
source_name                0
source_type                0
is_contaminated            0
communities_served         0
population_at_risk         0
treatment_plant_present    0
dtype: i

In [64]:
for t in tables:
    df = pd.read_sql(f"SELECT * FROM {t}", conn)

    print(f"\n {t} \n")
    print(df.describe())


 regions 

       region_id    population      area_km2  galamsey_hotspot
count  16.000000  1.600000e+01     16.000000         16.000000
mean    8.500000  1.793398e+06  19262.125000          0.500000
std     4.760952  1.604647e+06  15903.618703          0.516398
min     1.000000  3.620890e+05   3245.000000          0.000000
25%     4.750000  7.142055e+05  10498.750000          0.000000
50%     8.500000  1.190308e+06  13855.000000          0.500000
75%    12.250000  2.422823e+06  21407.750000          1.000000
max    16.000000  5.446789e+06  70384.000000          1.000000

 rivers 

        river_id  region_id    length_km  is_contaminated
count  38.000000  38.000000    38.000000        38.000000
mean   19.500000   8.026316   297.789474         0.473684
std    11.113055   4.795757   363.428729         0.506009
min     1.000000   1.000000    30.000000         0.000000
25%    10.250000   4.000000   100.000000         0.000000
50%    19.500000   8.000000   180.000000         0.000000
75% 

In [65]:
for t in tables:
    df = pd.read_sql(f"SELECT * FROM {t}", conn)

    print(f"\n {t} \n")
    df.info()


 regions 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   region_id         16 non-null     int64  
 1   region_name       16 non-null     object 
 2   capital           16 non-null     object 
 3   population        16 non-null     int64  
 4   area_km2          16 non-null     float64
 5   galamsey_hotspot  16 non-null     int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 900.0+ bytes

 rivers 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   river_id         38 non-null     int64  
 1   river_name       38 non-null     object 
 2   region_id        38 non-null     int64  
 3   length_km        38 non-null     float64
 4   flows_into       38 non-null     object 
 5   is_

# CELL 10 — CLEAN + ADD SAFETY FLAGS

In [66]:

df_wq = pd.read_sql("SELECT * FROM water_quality_readings", conn)
df_wq['reading_date'] = pd.to_datetime(df_wq['reading_date'])
df_wq['year'] = df_wq['reading_date'].dt.year

df_wq['mercury_unsafe']   = df_wq['mercury_ug_L']  > 0.001
df_wq['turbidity_unsafe'] = df_wq['turbidity_NTU'] > 2000.0
df_wq['cyanide_unsafe']   = df_wq['cyanide_mg_L']  > 0.07
df_wq['lead_unsafe']      = df_wq['lead_ug_L']     > 10.0
df_wq['ph_acidic']        = df_wq['ph_level']      < 6.5

print("=== UNSAFE READINGS SUMMARY ===")
print(f"Mercury above WHO limit:    {df_wq['mercury_unsafe'].sum()} / {len(df_wq)}")
print(f"Turbidity above GWCL limit: {df_wq['turbidity_unsafe'].sum()} / {len(df_wq)}")
print(f"Cyanide above WHO limit:    {df_wq['cyanide_unsafe'].sum()} / {len(df_wq)}")
print(f"Lead above WHO limit:       {df_wq['lead_unsafe'].sum()} / {len(df_wq)}")
print(f"pH below safe range:        {df_wq['ph_acidic'].sum()} / {len(df_wq)}")

=== UNSAFE READINGS SUMMARY ===
Mercury above WHO limit:    126 / 126
Turbidity above GWCL limit: 98 / 126
Cyanide above WHO limit:    94 / 126
Lead above WHO limit:       107 / 126
pH below safe range:        99 / 126


# CELL 11 — SQL JOINS: BUILD ANALYSIS DATAFRAMES

In [67]:
df_water_full =pd.read_sql("""
SELECT r.region_name, rv.river_name, rv.flows_into, wq.reading_date, wq.ph_level, wq.mercury_ug_L, wq.turbidity_NTU, wq.cyanide_mg_L, wq.lead_ug_L
FROM water_quality_readings AS wq
JOIN rivers AS rv
ON wq.river_id = rv.river_id
JOIN regions AS r
ON rv.region_id = r.region_id
ORDER BY r.region_name, wq.reading_date
""", conn)

df_water_full["reading_date"]=pd.to_datetime(df_water_full["reading_date"])
df_water_full["year"] = df_water_full["reading_date"].dt.year
print(f"✅ Water full join: {len(df_water_full)} rows")

✅ Water full join: 126 rows



# CELL 11 — SQL JOINS: BUILD ANALYSIS DATAFRAMES


In [68]:


# JOIN 1: Full water quality with region and river names
df_water_full = pd.read_sql("""
    SELECT r.region_name, rv.river_name, rv.flows_into,
           wq.reading_date, wq.ph_level, wq.mercury_ug_L,
           wq.turbidity_NTU, wq.cyanide_mg_L, wq.lead_ug_L
    FROM water_quality_readings wq
    JOIN rivers rv ON wq.river_id = rv.river_id
    JOIN regions r ON rv.region_id = r.region_id
    ORDER BY r.region_name, wq.reading_date
""", conn)
df_water_full['reading_date'] = pd.to_datetime(df_water_full['reading_date'])
df_water_full['year'] = df_water_full['reading_date'].dt.year
print(f"✅ Water full join: {len(df_water_full)} rows")


✅ Water full join: 126 rows


In [69]:

# JOIN 2: Mining with region names
df_mining_full = pd.read_sql("""
    SELECT r.region_name, ma.year, ma.mining_type,
           ma.hectares_destroyed, ma.num_sites, ma.chemicals_used
    FROM mining_activity ma
    JOIN regions r ON ma.region_id = r.region_id
    ORDER BY ma.year, ma.hectares_destroyed DESC
""", conn)
print(f"✅ Mining full join: {len(df_mining_full)} rows")



✅ Mining full join: 61 rows


In [70]:
# JOIN 3: Health with region names
df_health_full = pd.read_sql("""
    SELECT r.region_name, hi.year, hi.disease,
           hi.cases_reported, hi.deaths, hi.affected_children, hi.cause
    FROM health_impact hi
    JOIN regions r ON hi.region_id = r.region_id
    ORDER BY hi.year, hi.cases_reported DESC
""", conn)
print(f"✅ Health full join: {len(df_health_full)} rows")



✅ Health full join: 83 rows


In [71]:
# JOIN 4: Contaminated water supply with population at risk
df_supply_full = pd.read_sql("""
    SELECT r.region_name, ws.source_name, ws.source_type,
           ws.is_contaminated, ws.communities_served,
           ws.population_at_risk, ws.treatment_plant_present
    FROM water_supply_sources ws
    JOIN regions r ON ws.region_id = r.region_id
    ORDER BY ws.population_at_risk DESC
""", conn)
print(f"✅ Supply full join: {len(df_supply_full)} rows")



✅ Supply full join: 30 rows


In [72]:
# JOIN 5: MASTER — mercury + mining + health per region per year
df_master = pd.read_sql("""
    SELECT
        r.region_name,
        wq_agg.year,
        wq_agg.avg_mercury,
        wq_agg.avg_turbidity,
        wq_agg.avg_ph,
        ma_agg.total_hectares,
        ma_agg.total_sites,
        hi_agg.total_cases,
        hi_agg.total_deaths,
        hi_agg.total_children
    FROM regions r
    LEFT JOIN (
        SELECT rv.region_id,
               CAST(strftime('%Y', wq.reading_date) AS INTEGER) AS year,
               ROUND(AVG(wq.mercury_ug_L), 5)  AS avg_mercury,
               ROUND(AVG(wq.turbidity_NTU), 1) AS avg_turbidity,
               ROUND(AVG(wq.ph_level), 2)       AS avg_ph
        FROM water_quality_readings wq
        JOIN rivers rv ON wq.river_id = rv.river_id
        GROUP BY rv.region_id, year
    ) wq_agg ON r.region_id = wq_agg.region_id
    LEFT JOIN (
        SELECT region_id, year,
               ROUND(SUM(hectares_destroyed), 1) AS total_hectares,
               SUM(num_sites) AS total_sites
        FROM mining_activity
        GROUP BY region_id, year
    ) ma_agg ON r.region_id = ma_agg.region_id AND wq_agg.year = ma_agg.year
    LEFT JOIN (
        SELECT region_id, year,
               SUM(cases_reported)    AS total_cases,
               SUM(deaths)            AS total_deaths,
               SUM(affected_children) AS total_children
        FROM health_impact
        GROUP BY region_id, year
    ) hi_agg ON r.region_id = hi_agg.region_id AND wq_agg.year = hi_agg.year
    WHERE wq_agg.year IS NOT NULL
    ORDER BY r.region_name, wq_agg.year
""", conn)
df_master = df_master.dropna(subset=['avg_mercury'])
print(f"✅ Master join: {len(df_master)} rows")

✅ Master join: 81 rows


# CELL 12 — DOWNSTREAM CONTAMINATION RISK QUERY

In [73]:
df_connectivity = pd.read_sql("""
    SELECT
        r_up.river_name      AS upstream_river,
        reg_up.region_name   AS upstream_region,
        r_up.flows_into      AS flows_into,
        r_down.river_name    AS downstream_river,
        reg_down.region_name AS downstream_region,
        r_down.is_contaminated AS downstream_already_contaminated
    FROM rivers r_up
    JOIN rivers r_down ON r_up.flows_into = r_down.river_name
    JOIN regions reg_up   ON r_up.region_id   = reg_up.region_id
    JOIN regions reg_down ON r_down.region_id = reg_down.region_id
    WHERE r_up.is_contaminated = 1
    ORDER BY reg_up.region_name
""", conn)
print("=== DOWNSTREAM CONTAMINATION RISK ===")
print(df_connectivity.to_string())

=== DOWNSTREAM CONTAMINATION RISK ===
  upstream_river upstream_region  flows_into downstream_river downstream_region  downstream_already_contaminated
0      Oda River         Ashanti   Pra River        Pra River           Western                                1
1    Offin River         Ashanti   Pra River        Pra River           Western                                1
2    Owabi River         Ashanti   Pra River        Pra River           Western                                1
3    Subin River            Bono  Tano River       Tano River           Western                                1
4    Birim River         Eastern   Pra River        Pra River           Western                                1


# CELL 13 — VISUALIZATIONS

In [74]:
# PLOT 1: Mercury over time — hotspot rivers only
df_hotspot = df_water_full[df_water_full['region_name'].isin(
    ['Ashanti','Western','Eastern','Ahafo','Western North']
)]
fig1 = px.line(df_hotspot, x='reading_date', y='mercury_ug_L',
               color='river_name',
               title='Mercury Levels Over Time — Hotspot Rivers (2020–2026)',
               labels={'mercury_ug_L':'Mercury (µg/L)','reading_date':'Date'},
               markers=True)

fig1.show()


In [75]:
# PLOT 2: Pra River turbidity — the worst case
df_pra = df_water_full[df_water_full['river_name'] == 'Pra River']
fig2 = px.bar(df_pra, x='reading_date', y='turbidity_NTU',
              title='Pra River Turbidity (NTU) 2020–2026 — GWCL hit 14,000 NTU in 2024',
              color='turbidity_NTU', color_continuous_scale='Reds')
fig2.add_hline(y=2000, line_dash='dash', line_color='orange',
               annotation_text='GWCL Max Treatment Threshold (2000 NTU)')
fig2.show()


In [76]:
# PLOT 3: Hectares destroyed per region per year
fig3 = px.bar(df_mining_full, x='region_name', y='hectares_destroyed',
              color='year', barmode='group',
              title='Hectares Destroyed by Galamsey Per Region (2020–2026)')
fig3.show()

In [77]:
# PLOT 4: Health cases per region per year
df_cases = df_health_full.groupby(['region_name','year'])['cases_reported'].sum().reset_index()
fig4 = px.bar(df_cases, x='region_name', y='cases_reported',
              color='year', barmode='group',
              title='Total Health Cases Per Region Per Year (2020–2026)')
fig4.show()

In [78]:
# PLOT 5: Disease breakdown 2024 (most recent real data year)
df_2024 = df_health_full[df_health_full['year'] == 2024]
fig5 = px.bar(df_2024, x='region_name', y='cases_reported', color='disease',
              title='Disease Breakdown Per Region — 2024')
fig5.show()

In [79]:
import numpy as np
# PLOT 6: Mercury vs health cases (bubble = mining sites)
df_corr = df_master[df_master['total_cases'].notna()].copy()
# Fill NaN values in 'total_sites' with 0, as no mining sites would mean 0 sites.
df_corr['total_sites'] = df_corr['total_sites'].fillna(method = 'pad')
fig6 = px.scatter(df_corr, x='avg_mercury', y='total_cases',
                  size='total_sites', color='region_name',
                  facet_col='year',
                  title='Mercury Levels vs Health Cases — bubble size = mining sites',
                  hover_name='region_name')
fig6.show()

/tmp/ipykernel_9845/188414063.py:5: FutureWarning:

Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



In [80]:
# PLOT 7: Population at risk from contaminated sources
df_at_risk = df_supply_full[df_supply_full['is_contaminated'] == 1]
df_risk_grp = df_at_risk.groupby('region_name')['population_at_risk'].sum().reset_index()
df_risk_grp = df_risk_grp.sort_values('population_at_risk', ascending=False)
fig7 = px.bar(df_risk_grp, x='region_name', y='population_at_risk',
              title='Population at Risk from Contaminated Water Per Region',
              color='population_at_risk', color_continuous_scale='Reds')
fig7.show()

In [81]:
# Plot 9: Regions affected by galamsey vs not affected
df_regions_pie = pd.read_sql("SELECT galamsey_hotspot FROM regions", conn)
hotspot_counts = df_regions_pie['galamsey_hotspot'].value_counts().reset_index()
hotspot_counts.columns = ['status', 'count']
hotspot_counts['status'] = hotspot_counts['status'].map({1: 'Galamsey Hotspot', 0: 'Not Affected'})

fig_a = px.pie(
    hotspot_counts,
    names='status',
    values='count',
    title='Ghana Regions: Galamsey Hotspot vs Not Affected',
    color='status',
    color_discrete_map={'Galamsey Hotspot': 'red', 'Not Affected': 'green'}
)
fig_a.show()

In [82]:
# Plot 10: Contaminated rivers vs clean rivers across all of Ghana
df_rivers_pie = pd.read_sql("SELECT is_contaminated FROM rivers", conn)
river_counts = df_rivers_pie['is_contaminated'].value_counts().reset_index()
river_counts.columns = ['status', 'count']
river_counts['status'] = river_counts['status'].map({1: 'Contaminated', 0: 'Clean'})

fig_b = px.pie(
    river_counts,
    names='status',
    values='count',
    title='Ghana Rivers: Contaminated by Galamsey vs Clean',
    color='status',
    color_discrete_map={'Contaminated': 'red', 'Clean': 'green'}
)
fig_b.show()

In [83]:
# Plot 11: How many rivers per region are contaminated vs clean
df_rivers_region = pd.read_sql("""
    SELECT r.region_name,
           rv.is_contaminated
    FROM rivers rv
    JOIN regions r ON rv.region_id = r.region_id
""", conn)

df_rivers_region['status'] = df_rivers_region['is_contaminated'].map(
    {1: 'Contaminated', 0: 'Clean'}
)

df_stacked = df_rivers_region.groupby(
    ['region_name', 'status']
).size().reset_index(name='count')

fig_c = px.bar(
    df_stacked,
    x='region_name',
    y='count',
    color='status',
    barmode='stack',
    title='Number of Contaminated vs Clean Rivers Per Region',
    color_discrete_map={'Contaminated': 'red', 'Clean': 'green'},
    labels={'count': 'Number of Rivers', 'region_name': 'Region'}
)
fig_c.show()

In [84]:
# PLOT 12: Children affected over time
df_children = df_health_full.groupby(['region_name','year'])['affected_children'].sum().reset_index()
fig8 = px.line(df_children, x='year', y='affected_children',
               color='region_name', markers=True,
               title='Children Affected by Galamsey Health Impacts (2020–2026)')
fig8.show()


In [85]:
# PLOT 13: pH levels by region — acidification
fig9 = px.box(df_water_full, x='region_name', y='ph_level', color='region_name',
              title='River pH Distribution by Region (Safe range: 6.5–8.5)')
fig9.add_hline(y=6.5, line_dash='dash', line_color='red',
               annotation_text='Min Safe pH')
fig9.add_hline(y=8.5, line_dash='dash', line_color='blue',
               annotation_text='Max Safe pH')
fig9.show()

In [86]:
# PLOT 14: Deaths per region over time
df_deaths = df_health_full.groupby(['region_name','year'])['deaths'].sum().reset_index()
fig10 = px.line(df_deaths, x='year', y='deaths', color='region_name',
                markers=True,
                title='Deaths Linked to Galamsey Per Region (2020–2026)')
fig10.show()

In [87]:
# PLOT 15: Deaths per region over time
df_deaths = df_health_full.groupby(['region_name','year'])['deaths'].sum().reset_index()
fig10 = px.line(df_deaths, x='year', y='deaths', color='region_name',
                markers=True,
                title='Deaths Linked to Galamsey Per Region (2020–2026)')
fig10.show()

In [88]:
# PLOT 16: Total health cases per region 2020–2026 (all regions including zero)
df_all_regions = pd.read_sql("SELECT region_name FROM regions", conn)

df_total_cases = df_health_full.groupby('region_name')['cases_reported'].sum().reset_index()
df_total_cases.columns = ['region_name', 'total_cases']

# Merge so regions with zero cases still appear
df_total_cases = df_all_regions.merge(df_total_cases, on='region_name', how='left').fillna(0)
df_total_cases = df_total_cases.sort_values('total_cases', ascending=True)

fig_g = px.bar(
    df_total_cases,
    x='total_cases',
    y='region_name',
    orientation='h',
    title='Total Health Cases Linked to Galamsey Per Region (2020–2026)',
    labels={'total_cases': 'Total Cases', 'region_name': 'Region'},
    color='total_cases',
    color_continuous_scale='Reds'
)
fig_g.show()

In [89]:
# 3D SCATTER: Mercury level vs total health cases vs year per region
df_3d = df_master[df_master['total_cases'].notna()].copy()

fig_h = px.scatter_3d(
    df_3d,
    x='avg_mercury',
    y='total_cases',
    z='year',
    color='region_name',
    size='total_deaths',
    title='3D View: Mercury Levels vs Health Cases vs Year (size = deaths)',
    labels={
        'avg_mercury': 'Avg Mercury (µg/L)',
        'total_cases': 'Health Cases',
        'year': 'Year'
    },
    hover_name='region_name'
)
fig_h.show()

GALAMSEY IMPACT TRACKER — KEY INSIGHTS

In [90]:

# Most polluted river
mp = df_water_full.groupby('river_name')['mercury_ug_L'].mean()
print(f"\n1. Most polluted river: {mp.idxmax()}")
print(f"   Avg mercury: {mp.max():.5f} µg/L ({mp.max()/0.001:.0f}x above WHO limit)")



1. Most polluted river: Pra River
   Avg mercury: 0.01739 µg/L (17x above WHO limit)


In [91]:
# Region with most deaths
deaths = df_health_full.groupby('region_name')['deaths'].sum()
print(f"\n2. Most galamsey deaths: {deaths.idxmax()} ({deaths.max()} total 2020–2026)")



2. Most galamsey deaths: Ashanti (234 total 2020–2026)


In [92]:
# Total population at risk
at_risk = df_supply_full[df_supply_full['is_contaminated']==1]['population_at_risk'].sum()
print(f"\n3. Total population at risk: {at_risk:,}")



3. Total population at risk: 7,615,000


In [93]:
# Hectares destroyed by year
print(f"\n4. Hectares destroyed by year:")
for yr, ha in df_mining_full.groupby('year')['hectares_destroyed'].sum().items():
    print(f"   {yr}: {ha:.0f} ha")


4. Hectares destroyed by year:
   2020: 1475 ha
   2021: 1938 ha
   2022: 2513 ha
   2023: 3151 ha
   2024: 3918 ha
   2025: 4818 ha
   2026: 5717 ha


In [94]:
# Total children affected
print(f"\n5. Total children affected: {df_health_full['affected_children'].sum()}")


5. Total children affected: 4411


In [95]:
# Downstream risk
print(f"\n6. Downstream contamination risk:")
for _, row in df_connectivity.iterrows():
    print(f"   {row['upstream_river']} ({row['upstream_region']}) "
          f"→ flows into → {row['downstream_river']} ({row['downstream_region']})")


6. Downstream contamination risk:
   Oda River (Ashanti) → flows into → Pra River (Western)
   Offin River (Ashanti) → flows into → Pra River (Western)
   Owabi River (Ashanti) → flows into → Pra River (Western)
   Subin River (Bono) → flows into → Tano River (Western)
   Birim River (Eastern) → flows into → Pra River (Western)


In [96]:
# Mercury vs health correlation
corr = df_master[df_master['total_cases'].notna()]
if len(corr) > 3:
    r = corr['avg_mercury'].corr(corr['total_cases'])
    print(f"\n7. Mercury vs health cases correlation: {r:.3f}")


7. Mercury vs health cases correlation: 0.643


In [97]:
# DOWNSTREAM RISK: Which clean rivers are at risk from upstream contaminated rivers
df_downstream = pd.read_sql("""
    SELECT
        r_up.river_name      AS upstream_river,
        reg_up.region_name   AS upstream_region,
        r_up.flows_into      AS flows_into,
        r_down.river_name    AS downstream_river,
        reg_down.region_name AS downstream_region,
        CASE WHEN r_down.is_contaminated = 1
             THEN 'Already Contaminated'
             ELSE 'AT RISK — Not Yet Contaminated'
        END AS downstream_status
    FROM rivers r_up
    JOIN rivers r_down   ON r_up.flows_into    = r_down.river_name
    JOIN regions reg_up  ON r_up.region_id     = reg_up.region_id
    JOIN regions reg_down ON r_down.region_id  = reg_down.region_id
    WHERE r_up.is_contaminated = 1
    ORDER BY downstream_status DESC
""", conn)

print("=== DOWNSTREAM CONTAMINATION RISK ===")
print(df_downstream.to_string(index=False))
print()
print("INSIGHT:")
at_risk_rivers = df_downstream[df_downstream['downstream_status'] == 'AT RISK — Not Yet Contaminated']
already = df_downstream[df_downstream['downstream_status'] == 'Already Contaminated']
print(f"  {len(already)} downstream rivers are ALREADY contaminated from upstream flow.")
print(f"  {len(at_risk_rivers)} downstream rivers are currently CLEAN but AT RISK.")
for _, row in at_risk_rivers.iterrows():
    print(f"  ⚠️  {row['upstream_river']} ({row['upstream_region']}) is contaminated "
          f"and flows into {row['downstream_river']} ({row['downstream_region']}) — currently clean but at risk.")

=== DOWNSTREAM CONTAMINATION RISK ===
upstream_river upstream_region flows_into downstream_river downstream_region    downstream_status
   Offin River         Ashanti  Pra River        Pra River           Western Already Contaminated
     Oda River         Ashanti  Pra River        Pra River           Western Already Contaminated
   Owabi River         Ashanti  Pra River        Pra River           Western Already Contaminated
   Birim River         Eastern  Pra River        Pra River           Western Already Contaminated
   Subin River            Bono Tano River       Tano River           Western Already Contaminated

INSIGHT:
  5 downstream rivers are ALREADY contaminated from upstream flow.
  0 downstream rivers are currently CLEAN but AT RISK.


In [98]:
# BAR CHART: Downstream risk status
risk_counts = df_downstream['downstream_status'].value_counts().reset_index()
risk_counts.columns = ['status', 'count']

fig_e = px.bar(
    risk_counts,
    x='status',
    y='count',
    title='Downstream Rivers: Already Contaminated vs At Risk from Upstream Flow',
    color='status',
    color_discrete_map={
        'Already Contaminated': 'darkred',
        'AT RISK — Not Yet Contaminated': 'orange'
    },
    labels={'count': 'Number of Rivers', 'status': 'Contamination Status'}
)
fig_e.show()

# CELL 15 — CLOSING CONNECTION

In [99]:
conn.close()
print("✅ Connection closed.")

✅ Connection closed.
